In [1]:
# Install dependencies (run once)
%pip install shap missforest openpyxl joblib --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder,PowerTransformer,QuantileTransformer,MinMaxScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import r2_score, mean_squared_error
from missforest import MissForest
from sklearn.linear_model import LinearRegression
import shap as s
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import cross_val_predict
from scipy.optimize import differential_evolution
from scipy.spatial.distance import cdist
from itertools import combinations
import random
from joblib import Parallel, delayed
import multiprocessing
import optuna as op
# Reproducibility
RANDOM_STATE = 62

# Column config — centralised so changes propagate everywhere
TARGET_COL   = 'f_SAR_W_g'
CAT_COLS      = ['c_shape_type', 'c_magnetic_phase', 'c_coating']
DROP_COLS     = ['-', 'DOI', 'c_core_composition', 'c_dopant_element', 'f_dopant_ratio']
FEATURE_NAMES = [
    'f_avg_core_width_nm', 'f_Ms_emu_g_MNP', 'f_Hc_Oe', 'is_doped',
    'c_coating', 'f_H_Oe', 'f_FCY_kHz', 'f_conc_MNP_mg_mL',
    'c_shape_type', 'c_magnetic_phase', 'f_avg_core_sd_nm _BOTH',
    'Ca_comp', 'Co_comp', 'Cu_comp', 'Fe_comp', 'Ga_comp', 'Gd_comp',
    'Mg_comp', 'Mn_comp', 'Nd_comp', 'Ni_comp', 'Sn_comp', 'Tb_comp',
    'Zn_comp', 'f_magnetic_anisotropy', 'f_magnetic_dipole_moment',
]

In [3]:
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [
        c.replace('\ufeff', '').replace('\u200b', '').strip()
        for c in df.columns
    ]
    return df


df_raw = pd.read_excel("/kaggle/input/datasets/magneticmaterials/hyperthermia/Final.xlsx")
df_raw = clean_column_names(df_raw)

# Drop identifier/non-feature columns that exist in this dataset
cols_to_drop = [c for c in DROP_COLS if c in df_raw.columns]
df = df_raw.drop(columns=cols_to_drop)

print(f"Loaded {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

Loaded 1548 rows × 33 columns


,S. No.,f_avg_core_width_nm,f_Ms_emu_g_MNP,f_Hc_Oe,is_doped,f_is_coated,f_is_organic,f_is_inorganic,f_is_composite_coating,c_coating,...,Fe_comp,Ga_comp,Gd_comp,Mg_comp,Mn_comp,Nd_comp,Ni_comp,Sn_comp,Tb_comp,Zn_comp
0,1,15.0,60.00,0.0,0,0,0,0,0,NaN,...,3.00,0.0,0.00,0.0,0.0,0.0,0.0,0,0.0,0.0
1,2,17.0,36.78,6.0,1,1,1,0,0,Chitosan,...,2.93,0.0,0.07,0.0,0.0,0.0,0.0,0,0.0,0.0
2,3,75.0,60.00,100.0,0,0,0,0,0,NaN,...,3.00,0.0,0.00,0.0,0.0,0.0,0.0,0,0.0,0.0
3,4,19.9,53.10,4.0,0,1,1,0,0,TREG-coated,...,2.00,0.0,0.00,0.0,0.0,0.0,0.0,0,0.0,0.0
4,5,14.6,54.40,3.0,0,0,0,0,0,NaN,...,3.00,0.0,0.00,0.0,0.0,0.0,0.0,0,0.0,0.0


In [4]:
# --- 3e. Numeric feature correlations with target ---
numeric_df = df.select_dtypes(include=np.number)
if TARGET_COL in numeric_df.columns:
    corr = numeric_df.corr()[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False)
    print("Top correlations with target:")
    print(corr)

Top correlations with target:
f_Ms_emu_g_MNP                 0.309831
f_avg_core_width_nm            0.212310
f_area_volume_ratio           -0.157436
f_area_core_estimated_nm2      0.153174
f_H_Oe                         0.139183
f_FCY_kHz                      0.130932
Ca_comp                        0.108651
f_volume_core_estimated_nm3    0.108263
f_conc_MNP_mg_mL              -0.103827
S. No.                        -0.102738
f_is_inorganic                -0.071056
Cu_comp                        0.066102
f_is_organic                   0.053601
Tb_comp                        0.045630
f_Hc_Oe                       -0.041595
Mg_comp                       -0.037980
f_is_composite_coating         0.037516
Ga_comp                       -0.033765
Nd_comp                       -0.031919
Ni_comp                       -0.030614
Mn_comp                        0.029041
f_avg_core_sd_nm _BOTH        -0.025636
Sn_comp                       -0.024635
Co_comp                       -0.020375
f_is_coate

In [5]:
target_col = TARGET_COL.replace('\ufeff', '').replace('\u200b', '').strip()
X = df.drop(columns=[target_col])
y = df[target_col]

# 80 / 10 / 10 split
x_train, x_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE
)

print(f"Train : {x_train.shape[0]} rows")
print(f"Val   : {x_val.shape[0]} rows")
print(f"Test  : {x_test.shape[0]} rows  ← locked until final evaluation")

Train : 1238 rows
Val   : 155 rows
Test  : 155 rows  ← locked until final evaluation


In [6]:
# --- 5a. Binary presence indicators (fit-free, safe to apply before split logic) ---
def add_presence_flags(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    df_out['f_is_coated']  = df_out['c_coating'].notna().astype(int)
    df_out['f_is_shaped']  = df_out['c_shape_type'].notna().astype(int)
    df_out['f_is_phased']  = df_out['c_magnetic_phase'].notna().astype(int)
    return df_out

x_train = add_presence_flags(x_train)
x_val   = add_presence_flags(x_val)
x_test  = add_presence_flags(x_test)

In [7]:
# --- 5b. Categorical encoding ---
# Fill NaN → 'Unknown' for unseen categories, then ordinal-encode.
# Where the presence flag == 0 (field was missing), force the encoded value to -1.
# Encoders are fit on train only.

def encode_categorical(train_df, val_df, test_df, cat_col, flag_col):
    for df_part in [train_df, val_df, test_df]:
        df_part[cat_col] = df_part[cat_col].fillna('Unknown').astype(str)

    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    train_df[cat_col] = enc.fit_transform(train_df[[cat_col]])
    val_df[cat_col]   = enc.transform(val_df[[cat_col]])
    test_df[cat_col]  = enc.transform(test_df[[cat_col]])

    for df_part in [train_df, val_df, test_df]:
        df_part.loc[df_part[flag_col] == 0, cat_col] = -1
        df_part.drop(columns=[flag_col], inplace=True)

    return train_df, val_df, test_df

x_train, x_val, x_test = encode_categorical(x_train, x_val, x_test, 'c_coating',       'f_is_coated')
x_train, x_val, x_test = encode_categorical(x_train, x_val, x_test, 'c_shape_type',    'f_is_shaped')
x_train, x_val, x_test = encode_categorical(x_train, x_val, x_test, 'c_magnetic_phase', 'f_is_phased')

print("Encoding done. x_train shape:", x_train.shape)

Encoding done. x_train shape: (1238, 31)


In [8]:
# --- 5c. Numeric imputation (fit on train only) ---
numeric_cols = x_train.select_dtypes(include=np.number).columns.tolist()

imputer = MissForest(rgr=RandomForestRegressor(n_estimators=100,random_state=62),max_iter=10)

if x_train[numeric_cols].isnull().any().any():
    x_train[numeric_cols] = imputer.fit_transform(x_train[numeric_cols])
    x_val[numeric_cols]   = imputer.transform(x_val[numeric_cols])
    x_test[numeric_cols]  = imputer.transform(x_test[numeric_cols])
    print("Imputation applied.")
else:
    print("No missing values in numeric columns — skipping imputation.")

/usr/local/lib/python3.13/dist-packages/missforest/missforest.py:333: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
 30%|███       | 3/10 [01:13<02:53, 24.74s/it]/usr/local/lib/python3.13/dist-packages/missforest/missforest.py:303: UserWarning: NRMSE increased.
  warnings.warn("NRMSE increased.")
/usr/local/lib/python3.13/dist-packages/missforest/missforest.py:453: UserWarning: Stopping criterion triggered during fitting. Before last imputation matrix will be returned.
  warnings.warn(
 30%|███       | 3/10 [01:39<03:52, 33.27s/it]
/usr/local/lib/python3.13/dist-packages/missforest/missforest.py:490: UserWarning: Label encoding is no longer performed by default. Users will have to perform categorical features encoding by themselves.
  warnings.warn("Label encoding is no longer performed by default. "
/usr/local/lib/python3.13/dist

Imputation applied.


In [9]:
# --- 6a. Feature scaling (QuantileTransformer, fit on train only) ---
transf = QuantileTransformer(random_state=RANDOM_STATE)

x_train_scaled = transf.fit_transform(x_train)
x_val_scaled   = transf.transform(x_val)
x_test_scaled  = transf.transform(x_test)

print("Scaling done. Train shape:", x_train_scaled.shape)

Scaling done. Train shape: (1238, 31)


In [10]:
# --- 6b. Log transformation ---
y_train_log = np.log1p(y_train.values.astype(float))   # log1p on raw SAR
y_val_log   = np.log1p(y_val.values.astype(float))
y_test_log  = np.log1p(y_test.values.astype(float))

scaler = MinMaxScaler()
y_train_scaled = scaler.fit_transform(y_train_log.reshape(-1, 1)).flatten()
y_val_scaled   = scaler.transform(y_val_log.reshape(-1, 1)).flatten()
y_test_scaled  = scaler.transform(y_test_log.reshape(-1, 1)).flatten()

print("=== Scaler sanity check ===")
print(f"  data_min_ : {scaler.data_min_[0]:.4f}  "
      f"(= log1p({np.expm1(scaler.data_min_[0]):.2f} W/g)")
print(f"  data_max_ : {scaler.data_max_[0]:.4f}  "
      f"(= log1p({np.expm1(scaler.data_max_[0]):.2f} W/g)")
assert scaler.data_max_[0] > 1.0, \
    "Scaler data_max_ is still wrong — y_train may still be pre-transformed. Restart kernel and rerun top-to-bottom."

# Use these as your targets for model training (rename to match rest of notebook)
y_train = y_train_scaled
y_val   = y_val_scaled
y_test  = y_test_scaled


=== Scaler sanity check ===
  data_min_ : 0.1570  (= log1p(0.17 W/g)
  data_max_ : 8.6145  (= log1p(5510.00 W/g)


In [11]:
def inverse_sar(y_scaled: np.ndarray, fitted_scaler: MinMaxScaler) -> np.ndarray:
  y_scaled_clipped = np.clip(y_scaled, 0.0, 1.0)
  pred_log_units      = fitted_scaler.inverse_transform(y_scaled_clipped.reshape(-1, 1)).flatten()
  true_predicted_sar  = np.expm1(pred_log_units)
  return true_predicted_sar

In [12]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ConstantKernel as C, WhiteKernel
from sklearn.metrics import root_mean_squared_error
import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress GPR convergence warnings during Optuna trial exploration
warnings.filterwarnings("ignore", category=ConvergenceWarning)

def build_gpr_from_params(params: dict) -> GaussianProcessRegressor:
    """Helper to construct the GPR model and composite kernel from a parameter dictionary."""
    kernel_type  = params['kernel_type']
    const_val    = params['constant_value']
    length_scale = params['length_scale']
    noise_level  = params['noise_level']
    alpha        = params['alpha']
    normalize_y  = params['normalize_y']

    if kernel_type == 'RBF':
        base_kernel = RBF(length_scale=length_scale, length_scale_bounds=(1e-3, 1e3))
    elif kernel_type == 'Matern':
        nu = params['matern_nu']
        base_kernel = Matern(length_scale=length_scale, nu=nu, length_scale_bounds=(1e-3, 1e3))
    elif kernel_type == 'RationalQuadratic':
        rq_alpha = params['rq_alpha']
        base_kernel = RationalQuadratic(
            length_scale=length_scale, alpha=rq_alpha,
            length_scale_bounds=(1e-3, 1e3), alpha_bounds=(1e-3, 1e3)
        )

    # Composite kernel: Constant scaling * Base Kernel + WhiteNoise
    kernel = (
        C(const_val, (1e-3, 1e3)) * base_kernel 
        + WhiteKernel(noise_level=noise_level, noise_level_bounds=(1e-6, 1e1))
    )

    return GaussianProcessRegressor(
        kernel=kernel,
        alpha=alpha,
        normalize_y=normalize_y,
        n_restarts_optimizer=3,
        random_state=RANDOM_STATE
    )

# Pre-compute true validation SAR once for consistent RMSE comparison
y_val_true_sar = inverse_sar(y_val, scaler)

def objective(trial):
    kernel_type = trial.suggest_categorical('kernel_type', ['RBF', 'Matern', 'RationalQuadratic'])
    
    params = {
        'kernel_type':    kernel_type,
        'constant_value': trial.suggest_float('constant_value', 0.1, 10.0, log=True),
        'length_scale':   trial.suggest_float('length_scale', 0.1, 10.0, log=True),
        'noise_level':    trial.suggest_float('noise_level', 1e-5, 1e-1, log=True),
        'alpha':          trial.suggest_float('alpha', 1e-6, 1e-1, log=True),
        'normalize_y':    trial.suggest_categorical('normalize_y', [True, False]),
    }
    
    if kernel_type == 'Matern':
        params['matern_nu'] = trial.suggest_categorical('matern_nu', [0.5, 1.5, 2.5])
    elif kernel_type == 'RationalQuadratic':
        params['rq_alpha'] = trial.suggest_float('rq_alpha', 0.1, 10.0, log=True)

    gpr = build_gpr_from_params(params)
    gpr.fit(x_train_scaled, y_train)
    
    preds_scaled = gpr.predict(x_val_scaled)
    preds_sar    = inverse_sar(preds_scaled, scaler)
    
    rmse = root_mean_squared_error(y_val_true_sar, preds_sar)
    return rmse

study = op.create_study(
    direction='minimize',
    sampler=op.samplers.TPESampler(seed=RANDOM_STATE),
    storage='sqlite:///gpr_optuna.db',
    load_if_exists=True,
)

study.optimize(objective, n_trials=20)

print("Best params :", study.best_params)
print("Best val RMSE (W/g):", round(study.best_value, 4))

[I 2026-10-21 06:10:27,354] A new study created in RDB with name: no-name-f1653906-ca08-49f3-a2db-a2290dc0ed7e
[I 2026-10-21 06:11:46,257] Trial 0 finished with value: 203.82979655397156 and parameters: {'kernel_type': 'RationalQuadratic', 'constant_value': 0.6649729811414404, 'length_scale': 1.8316004749057304, 'noise_level': 0.0001458051785529714, 'alpha': 0.0005176085537373429, 'normalize_y': False, 'rq_alpha': 8.07077154474988}. Best is trial 0 with value: 203.82979655397156.
[I 2026-10-21 06:12:55,915] Trial 1 finished with value: 204.05527400742392 and parameters: {'kernel_type': 'RationalQuadratic', 'constant_value': 2.953854295346235, 'length_scale': 2.77897680384984, 'noise_level': 0.00043851659489621374, 'alpha': 1.044482010094054e-05, 'normalize_y': True, 'rq_alpha': 0.12461362524229155}. Best is trial 0 with value: 203.82979655397156.
[I 2026-10-21 06:14:13,948] Trial 2 finished with value: 204.05527223058618 and parameters: {'kernel_type': 'RationalQuadratic', 'constant_va

Best params : {'kernel_type': 'RBF', 'constant_value': 1.9289755372313102, 'length_scale': 7.2616655518105055, 'noise_level': 1.8076859490861418e-05, 'alpha': 0.004428174549911193, 'normalize_y': False}
Best val RMSE (W/g): 198.3096


In [13]:
gpr = build_gpr_from_params(study.best_params)
gpr.n_restarts_optimizer = 5  # Extra restarts for the final production model
gpr.fit(x_train_scaled, y_train)

print("Learned Kernel:", gpr.kernel_)

Learned Kernel: 0.29**2 * RBF(length_scale=1.2) + WhiteKernel(noise_level=1e-06)


In [14]:
# Predict with uncertainty (std in scaled space)
y_val_pred, y_val_std   = gpr.predict(x_val_scaled, return_std=True)
y_test_pred, y_test_std = gpr.predict(x_test_scaled, return_std=True)

# Metrics in scaled [0, 1] space
val_r2    = r2_score(y_val,  y_val_pred)
test_r2   = r2_score(y_test, y_test_pred)
val_rmse  = np.sqrt(mean_squared_error(y_val, y_val_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_q90  = np.quantile(np.abs(y_test - y_test_pred), 0.90)

# Inverse transform to original SAR (W/g) units without overwriting y_test
y_test_orig      = inverse_sar(y_test, scaler)
y_test_pred_orig = inverse_sar(y_test_pred, scaler)
err_90           = np.quantile(np.abs(y_test_orig - y_test_pred_orig), 0.90)

print(f"Validation  — R²: {val_r2:.4f}   RMSE (scaled): {val_rmse:.4f}")
print(f"Test        — R²: {test_r2:.4f}   RMSE (scaled): {test_rmse:.4f}")
print(f"90th-pct prediction error (test, W/g): ±{err_90:.4f}")
print(f"Mean GPR predictive std (scaled test): {np.mean(y_test_std):.4f}")

Validation  — R²: 0.6535   RMSE (scaled): 0.0986
Test        — R²: 0.6797   RMSE (scaled): 0.0903
90th-pct prediction error (test, W/g): ±135.1763
Mean GPR predictive std (scaled test): 0.0545


In [15]:
# 1. Actual vs Predicted Parity Plot
plt.figure(figsize=(8, 6))
plt.scatter(
    y_test_orig, y_test_pred_orig, 
    alpha=0.6, color='teal', edgecolor='k', label='GPR Predictions'
)
min_val = min(np.min(y_test_orig), np.min(y_test_pred_orig))
max_val = max(np.max(y_test_orig), np.max(y_test_pred_orig))
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal (Actual = Predicted)')
plt.title('Gaussian Process Regression: Actual vs Predicted SAR', fontsize=14)
plt.xlabel('Actual SAR (W/g)', fontsize=12)
plt.ylabel('Predicted SAR (W/g)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('gpr_actual_vs_predicted.png', dpi=600, bbox_inches='tight')
plt.close()

# 2. SHAP Summary Plot using KernelExplainer + K-Means background summary
background_summary = s.kmeans(x_train_scaled, 50)
explainer = s.KernelExplainer(gpr.predict, background_summary)
shap_values = explainer.shap_values(x_test_scaled, nsamples=200)

plt.figure(figsize=(10, 8))
s.summary_plot(shap_values, x_test_scaled, feature_names=x_test.columns, show=False)
plt.title('Gaussian Process Regression: SHAP Summary', fontsize=14)
plt.tight_layout()
plt.savefig('gpr_shap_summary.png', dpi=600, bbox_inches='tight')
plt.close()

  0%|          | 0/155 [00:00<?, ?it/s]